# MITONet - Shinnecock Inlet

In [ ]:
import os
import sys
import tensorflow as tf
tf.keras.backend.set_floatx('float64') 

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from matplotlib.ticker import LinearLocator, ScalarFormatter, FormatStrFormatter
from matplotlib import ticker as mtick
from matplotlib.patches import Patch
import cmocean

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error as mae
import joblib

from IPython.display import display
import matplotlib.ticker as ticker
from matplotlib import rcParams
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import netCDF4 as nc
from tqdm import tqdm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches


plt.rc('font', family='serif')
plt.rcParams.update({'font.size': 20,
                     'lines.linewidth': 2,
                     'lines.markersize':10,
                     'axes.labelsize': 16, # fontsize for x and y labels (was 10)
                     'axes.titlesize': 20,
                     'xtick.labelsize': 16,
                     'ytick.labelsize': 16,
                     'legend.fontsize': 16,
                     'axes.linewidth': 2})

import itertools
colors = itertools.cycle(['r','g','b','m','y','c'])
markers = itertools.cycle(['p','d','o','^','s','x',]) #'D','H','v','*'])

from pathlib import Path
try:
    work_dir.exists()
except NameError:
    curr_dir = Path().resolve()
    work_dir = curr_dir.parent  

scripts_dir = work_dir / "src"
sys.path.append(str(scripts_dir.absolute()))

# ae_model_dir = work_dir / "saved_models"
# mito_model_dir = work_dir / "saved_models"
# scalers_dir = work_dir / "scalers"
ae_model_dir = work_dir / "saved_models_mito"
mito_model_dir = work_dir / "saved_models_mito"
scalers_dir = work_dir / "scalers_mito"

#Download and copy your data to this folder
ROOT_DIR = work_dir.parents[1]
SHINNECOCK_DATA_DIR = (
    ROOT_DIR
    / 'data'
    / 'PRJ-5716'
    / 'Simulation--2d-adcirc-simulation-of-tidal-flow-in-shinnecock-inlet-ny-parameterized-by-bottom-friction-coefficient'
    / 'data'
    / 'Model--adcirc-model'
    / 'data'
)
data_dir = SHINNECOCK_DATA_DIR


import mitonet as mito
import autoencoder as ae
import data_loader as dl 
import processing_utils as pu

from importlib import reload as reload

%matplotlib inline

## Specify training configuraiton for scaling purposes and testing cases for visualization

In [ ]:
# Training data for scaling
train_days = [15., 30.]
param_list_train = [0.003, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
scaling = True

# Test data
test_days = [5., 45.]
param_list_test = [0.0025, 0.00275, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.075, 0.1]
# unseen_param_list_test = [0.0025, 0.015, 0.1]
unseen_param_loc = [0,6,-1]


## Load and visualize mesh, bathymetry and sensor locations

In [ ]:
# Load mesh
mesh = dl.load_mesh(data_dir/"cf0025")

# Extract fields
field = mesh['bathy']
triangles = mesh['triangles'] - 1  # if your indexing in 'triangles' starts at 1, convert to 0-based
lon = mesh['nodes'][:, 0]
lat = mesh['nodes'][:, 1]

# Sensors (node indices)
sensors = [2775, 2624, 1115]
s1_color = cmocean.cm.phase(0.6)
s2_color = cmocean.cm.phase(1)
s3_color = cmocean.cm.phase(0.8)

# Compute valmin and valmax from valid (unmasked) data
valmax = field[~field.mask].max()
valmin = field[~field.mask].min()

# ---- Single figure with two subplots ----
fig = plt.figure(figsize=(30,20))

# LEFT: full domain
ax = plt.subplot(1, 2, 1, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
wmts_url = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/WMTS/1.0.0/WMTSCapabilities.xml"
ax.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry
mesh_plot_left = ax.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent = [-73.0, -71.95, 40.3, 41.05]
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Customize tick labels
ax.set_xticks([-73, -72.5, -72], crs=ccrs.PlateCarree())
ax.set_yticks([40.4, 40.6, 40.8, 41.0], crs=ccrs.PlateCarree())
lon_formatter = LongitudeFormatter(degree_symbol='°')
lat_formatter = LatitudeFormatter(degree_symbol='°')
ax.xaxis.set_major_formatter(lon_formatter)
ax.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers (noticeable, non-circular)
ax.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)
ax.legend(loc='upper left')

# RIGHT: zoomed-in
ax2 = plt.subplot(1, 2, 2, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
ax2.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry (same vmin/vmax for shared colorbar)
mesh_plot_right = ax2.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent_zoom = [-72.55, -72.4, 40.8, 40.9]
ax2.set_extent(extent_zoom, crs=ccrs.PlateCarree())

# Customize tick labels
ax2.set_xticks([-72.54, -72.48, -72.4], crs=ccrs.PlateCarree())
ax2.set_yticks([40.8, 40.85, 40.9], crs=ccrs.PlateCarree())
ax2.xaxis.set_major_formatter(lon_formatter)
ax2.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers to zoom panel (same markers/colors)
ax2.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)

# Single colorbar on the right (attach to both axes)
cbar = fig.colorbar(mesh_plot_right, ax=[ax, ax2], shrink=0.4, pad=0.04)
cbar.set_label('Depth (m)')

plt.show()


## Load test data

In [ ]:
Np = len(param_list_test)

mesh = dl.load_mesh(data_dir/"cf0025")
snaps = [np.where(mesh['t']/60/60/24==test_days[0])[0][0], np.where(mesh['t']/60/60/24==test_days[1])[0][0]]
train_snaps = [np.where(mesh['t']/60/60/24==train_days[0])[0][0], np.where(mesh['t']/60/60/24==train_days[1])[0][0]]
Nt_snaps = mesh['t'][snaps[0]:snaps[1]].shape[0]

data_dict = {}
for inx, param in enumerate(param_list_test):
    dirname = 'cf'+str(param).split('.')[1];
    data_dict[inx] = dl.load_variables(data_dir/dirname)

Nn = mesh['nodes'].shape[0]
Ne = mesh['triangles'].shape[0]
Nt = mesh['t'].shape[0]


In [ ]:
dataset_h = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_u = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_v = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed

key = 'h'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_h = np.vstack((dataset_h, snap))

key = 'u'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_u = np.vstack((dataset_u, snap))

key = 'v'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_v = np.vstack((dataset_v, snap))
    
dataset_u_resh = np.reshape(dataset_u,(Np,Nt,Nn))
dataset_u_crop = dataset_u_resh[:,snaps[0]:snaps[1],:]
dataset_u = np.reshape(dataset_u_crop,(Np*Nt_snaps,Nn))

dataset_h_resh = np.reshape(dataset_h,(Np,Nt,Nn))
dataset_h_crop = dataset_h_resh[:,snaps[0]:snaps[1],:]
dataset_h = np.reshape(dataset_h_crop,(Np*Nt_snaps,Nn))

dataset_v_resh = np.reshape(dataset_v,(Np,Nt,Nn))
dataset_v_crop = dataset_v_resh[:,snaps[0]:snaps[1],:]
dataset_v = np.reshape(dataset_v_crop,(Np*Nt_snaps,Nn))

dataset_r_h = np.reshape(dataset_h,(Np,Nt_snaps,Nn))
dataset_r_u = np.reshape(dataset_u,(Np,Nt_snaps,Nn))
dataset_r_v = np.reshape(dataset_v,(Np,Nt_snaps,Nn))


### Load Predictions

In [ ]:
mito_data = np.load('mito_output.npz')
H_mito = mito_data['h_data']
U_mito = mito_data['u_data']
V_mito = mito_data['v_data']

don_data = np.load('don_output.npz')
H_don = don_data['h_data']
U_don = don_data['u_data']
V_don = don_data['v_data']

ldon_data = np.load('ldon_output.npz')
H_ldon = ldon_data['h_data']
U_ldon = ldon_data['u_data']
V_ldon = ldon_data['v_data']

mdon_data = np.load('mdon_output.npz')
H_mdon = mdon_data['h_data']
U_mdon = mdon_data['u_data']
V_mdon = mdon_data['v_data']

mion_data = np.load('mion_output.npz')
H_mion = mion_data['h_data']
U_mion = mion_data['u_data']
V_mion = mion_data['v_data']

#### Contour Plots

In [ ]:
x_train_lower = mesh['t'][train_snaps[0]]/3600/24
x_train_upper = mesh['t'][train_snaps[1]]/3600/24

x_lower = mesh['t'][snaps[0]]/3600/24 
x_upper = mesh['t'][snaps[1]]/3600/24
time = mesh['t'][snaps[0]:snaps[1]]/60/60/24

colormap = [cmocean.cm.balance(0.375), cmocean.cm.balance(0.25), cmocean.cm.balance(0.125)]

z_order=[2,0,1]

In [ ]:
tstep = -1
param1 = -1
param2 = 0
print(time[tstep])

fig, axs = plt.subplots(6, 3, figsize=(15, 20), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,0].axis('off')

axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mito[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,0].axis('off')

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_don[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,0].axis('off')

axs[3,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mdon[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,0].axis('off')

axs[4,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_ldon[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,0].axis('off')

last_h = axs[5,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mion[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,0].axis('off')

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_u[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mito[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_don[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,1].axis('off')

axs[3,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mdon[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,1].axis('off')

axs[4,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_ldon[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,1].axis('off')

last_u = axs[5,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mion[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,1].axis('off')

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_v[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mito[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_don[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,2].axis('off')

axs[3,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mdon[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,2].axis('off')

axs[4,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_ldon[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,2].axis('off')

last_v = axs[5,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mion[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,2].axis('off')

axs[0,0].set_title('r: 0.1')
axs[0,1].set_title('r: 0.0025')
axs[0,2].set_title('r: 0.0025')

axs[0, 0].text(-0.05, 0.5, 'ADCIRC', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.05, 0.5, 'DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)
axs[3, 0].text(-0.05, 0.5, 'M-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[3,0].transAxes)
axs[4, 0].text(-0.05, 0.5, 'L-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[4,0].transAxes)
axs[5, 0].text(-0.05, 0.5, 'MIONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[5,0].transAxes)


cbar = plt.colorbar(last_h, orientation='horizontal', aspect=18)
cbar.set_label(label='Water Surface Elevation (cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='horizontal', aspect=18)
cbar.set_label(label='U-Velocity (cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='horizontal', aspect=18)
cbar.set_label(label='V-Velocity (cm/s)',  size=20)  

In [ ]:
tstep = -1
param1 = -1
param2 = 0
print(time[tstep])

fig, axs = plt.subplots(6, 3, figsize=(15, 20), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,0].axis('off')
axs[0,0].set_xlim([-72.50, -72.46])
axs[0,0].set_ylim([40.82,40.88]) 

axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mito[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,0].axis('off')
axs[1,0].set_xlim([-72.50, -72.46])
axs[1,0].set_ylim([40.82,40.88]) 

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_don[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,0].axis('off')
axs[2,0].set_xlim([-72.50, -72.46])
axs[2,0].set_ylim([40.82,40.88]) 

axs[3,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mdon[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,0].axis('off')
axs[3,0].set_xlim([-72.50, -72.46])
axs[3,0].set_ylim([40.82,40.88]) 

axs[4,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_ldon[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,0].axis('off')
axs[4,0].set_xlim([-72.50, -72.46])
axs[4,0].set_ylim([40.82,40.88]) 

last_h = axs[5,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   H_mion[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,0].axis('off')
axs[5,0].set_xlim([-72.50, -72.46])
axs[5,0].set_ylim([40.82,40.88]) 

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_u[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')
axs[0,1].set_xlim([-72.50, -72.46])
axs[0,1].set_ylim([40.82,40.88]) 

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mito[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')
axs[1,1].set_xlim([-72.50, -72.46])
axs[1,1].set_ylim([40.82,40.88]) 

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_don[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,1].axis('off')
axs[2,1].set_xlim([-72.50, -72.46])
axs[2,1].set_ylim([40.82,40.88]) 

axs[3,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mdon[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,1].axis('off')
axs[3,1].set_xlim([-72.50, -72.46])
axs[3,1].set_ylim([40.82,40.88]) 

axs[4,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_ldon[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,1].axis('off')
axs[4,1].set_xlim([-72.50, -72.46])
axs[4,1].set_ylim([40.82,40.88]) 

last_u = axs[5,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   U_mion[param1,tstep,:]*100, vmin=dataset_r_u[param1,tstep,:].min()*100, 
                   vmax=dataset_r_u[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,1].axis('off')
axs[5,1].set_xlim([-72.50, -72.46])
axs[5,1].set_ylim([40.82,40.88]) 

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_v[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')
axs[0,2].set_xlim([-72.50, -72.46])
axs[0,2].set_ylim([40.82,40.88]) 

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mito[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')
axs[1,2].set_xlim([-72.50, -72.46])
axs[1,2].set_ylim([40.82,40.88]) 

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_don[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,2].axis('off')
axs[2,2].set_xlim([-72.50, -72.46])
axs[2,2].set_ylim([40.82,40.88]) 

axs[3,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mdon[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,2].axis('off')
axs[3,2].set_xlim([-72.50, -72.46])
axs[3,2].set_ylim([40.82,40.88]) 

axs[4,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_ldon[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[4,2].axis('off')
axs[4,2].set_xlim([-72.50, -72.46])
axs[4,2].set_ylim([40.82,40.88]) 

last_v = axs[5,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   V_mion[param1,tstep,:]*100, vmin=dataset_r_v[param1,tstep,:].min()*100, 
                   vmax=dataset_r_v[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[5,2].axis('off')
axs[5,2].set_xlim([-72.50, -72.46])
axs[5,2].set_ylim([40.82,40.88]) 

axs[0,0].set_title('r: 0.1')
axs[0,1].set_title('r: 0.0025')
axs[0,2].set_title('r: 0.0025')

axs[0, 0].text(-0.05, 0.5, 'ADCIRC', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.05, 0.5, 'DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)
axs[3, 0].text(-0.05, 0.5, 'M-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[3,0].transAxes)
axs[4, 0].text(-0.05, 0.5, 'L-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[4,0].transAxes)
axs[5, 0].text(-0.05, 0.5, 'MIONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[5,0].transAxes)


cbar = plt.colorbar(last_h, orientation='horizontal', aspect=18)
cbar.set_label(label='Water Surface Elevation (cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='horizontal', aspect=18)
cbar.set_label(label='U-Velocity (cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='horizontal', aspect=18)
cbar.set_label(label='V-Velocity (cm/s)',  size=20)  

In [ ]:
param1 = -1
param2 = 0

# Precompute the two error fields:
err_h_don = np.mean(np.abs(dataset_r_h[param2, :, :] - H_don[param2, :, :]),axis=0)
err_u_don = np.mean(np.abs(dataset_r_u[param1, :, :] - U_don[param1, :, :]),axis=0)
err_v_don = np.mean(np.abs(dataset_r_v[param1, :, :] - V_don[param1, :, :]),axis=0)

err_h_mdon = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mdon[param2, :, :]),axis=0)
err_u_mdon = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mdon[param1, :, :]),axis=0)
err_v_mdon = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mdon[param1, :, :]),axis=0)

err_h_ldon = np.mean(np.abs(dataset_r_h[param2, :, :] - H_ldon[param2, :, :]),axis=0)
err_u_ldon = np.mean(np.abs(dataset_r_u[param1, :, :] - U_ldon[param1, :, :]),axis=0)
err_v_ldon = np.mean(np.abs(dataset_r_v[param1, :, :] - V_ldon[param1, :, :]),axis=0)

err_h_mion = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mion[param2, :, :]),axis=0)
err_u_mion = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mion[param1, :, :]),axis=0)
err_v_mion = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mion[param1, :, :]),axis=0)

err_h_mito = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mito[param2, :, :]),axis=0)
err_u_mito = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mito[param1, :, :]),axis=0)
err_v_mito = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mito[param1, :, :]),axis=0)

fig, axs = plt.subplots(5, 3, figsize=(15,18), layout='constrained')#, sharey='col')
# axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, full_res_r_h[0,0,:], vmin=-0.5, vmax=0.5, cmap='cmo.amp', shading='gouraud')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mito*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,0].axis('off')


axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_don*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,0].axis('off')


axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mdon*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,0].axis('off')


axs[3,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_ldon*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,0].axis('off')


last_h = axs[4,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mion*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,0].axis('off')


axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mito*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,1].axis('off')


axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_don*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,1].axis('off')


axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mdon*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,1].axis('off')


axs[3,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_ldon*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,1].axis('off')


last_u = axs[4,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mion*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,1].axis('off')


axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mito*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,2].axis('off')


axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_don*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,2].axis('off')


axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mdon*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,2].axis('off')


axs[3,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_ldon*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,2].axis('off')


last_v = axs[4,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mion*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,2].axis('off')


axs[0, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.05, 0.5, 'M-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)
axs[3, 0].text(-0.05, 0.5, 'L-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[3,0].transAxes)
axs[4, 0].text(-0.05, 0.5, 'MIONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[4,0].transAxes)

axs[0, 0].set_title('Water Surface Elevation (r: 0.1)', size=20,transform=axs[0,0].transAxes)
axs[0, 1].set_title('U-Velocity (r: 0.0025)', size=20,transform=axs[0,1].transAxes)
axs[0, 2].set_title('V-Velocity (r: 0.0025)', size=20,transform=axs[0,2].transAxes)


cbar = plt.colorbar(last_h, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)',  size=20) 

In [ ]:
param1 = -1
param2 = 0

# Precompute the two error fields:
err_h_don = np.mean(np.abs(dataset_r_h[param2, :, :] - H_don[param2, :, :]),axis=0)
err_u_don = np.mean(np.abs(dataset_r_u[param1, :, :] - U_don[param1, :, :]),axis=0)
err_v_don = np.mean(np.abs(dataset_r_v[param1, :, :] - V_don[param1, :, :]),axis=0)

err_h_mdon = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mdon[param2, :, :]),axis=0)
err_u_mdon = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mdon[param1, :, :]),axis=0)
err_v_mdon = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mdon[param1, :, :]),axis=0)

err_h_ldon = np.mean(np.abs(dataset_r_h[param2, :, :] - H_ldon[param2, :, :]),axis=0)
err_u_ldon = np.mean(np.abs(dataset_r_u[param1, :, :] - U_ldon[param1, :, :]),axis=0)
err_v_ldon = np.mean(np.abs(dataset_r_v[param1, :, :] - V_ldon[param1, :, :]),axis=0)

err_h_mion = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mion[param2, :, :]),axis=0)
err_u_mion = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mion[param1, :, :]),axis=0)
err_v_mion = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mion[param1, :, :]),axis=0)

err_h_mito = np.mean(np.abs(dataset_r_h[param2, :, :] - H_mito[param2, :, :]),axis=0)
err_u_mito = np.mean(np.abs(dataset_r_u[param1, :, :] - U_mito[param1, :, :]),axis=0)
err_v_mito = np.mean(np.abs(dataset_r_v[param1, :, :] - V_mito[param1, :, :]),axis=0)

fig, axs = plt.subplots(5, 3, figsize=(15,18), layout='constrained')#, sharey='col')
# axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, full_res_r_h[0,0,:], vmin=-0.5, vmax=0.5, cmap='cmo.amp', shading='gouraud')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mito*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,0].axis('off')
axs[0,0].set_xlim([-72.50, -72.46])
axs[0,0].set_ylim([40.82,40.88]) 

axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_don*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,0].axis('off')
axs[1,0].set_xlim([-72.50, -72.46])
axs[1,0].set_ylim([40.82,40.88]) 

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mdon*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,0].axis('off')
axs[2,0].set_xlim([-72.50, -72.46])
axs[2,0].set_ylim([40.82,40.88]) 

axs[3,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_ldon*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,0].axis('off')
axs[3,0].set_xlim([-72.50, -72.46])
axs[3,0].set_ylim([40.82,40.88]) 

last_h = axs[4,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_h_mion*100, vmin=err_h_don.min()*100, 
                   vmax=err_h_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,0].axis('off')
axs[4,0].set_xlim([-72.50, -72.46])
axs[4,0].set_ylim([40.82,40.88]) 

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mito*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,1].axis('off')
axs[0,1].set_xlim([-72.50, -72.46])
axs[0,1].set_ylim([40.82,40.88]) 

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_don*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,1].axis('off')
axs[1,1].set_xlim([-72.50, -72.46])
axs[1,1].set_ylim([40.82,40.88]) 

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mdon*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,1].axis('off')
axs[2,1].set_xlim([-72.50, -72.46])
axs[2,1].set_ylim([40.82,40.88]) 

axs[3,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_ldon*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,1].axis('off')
axs[3,1].set_xlim([-72.50, -72.46])
axs[3,1].set_ylim([40.82,40.88]) 

last_u = axs[4,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_u_mion*100, vmin=err_u_don.min()*100, 
                   vmax=err_u_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,1].axis('off')
axs[4,1].set_xlim([-72.50, -72.46])
axs[4,1].set_ylim([40.82,40.88]) 

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mito*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[0,2].axis('off')
axs[0,2].set_xlim([-72.50, -72.46])
axs[0,2].set_ylim([40.82,40.88]) 

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_don*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[1,2].axis('off')
axs[1,2].set_xlim([-72.50, -72.46])
axs[1,2].set_ylim([40.82,40.88]) 

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mdon*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[2,2].axis('off')
axs[2,2].set_xlim([-72.50, -72.46])
axs[2,2].set_ylim([40.82,40.88]) 

axs[3,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_ldon*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[3,2].axis('off')
axs[3,2].set_xlim([-72.50, -72.46])
axs[3,2].set_ylim([40.82,40.88]) 

last_v = axs[4,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   err_v_mion*100, vmin=err_v_don.min()*100, 
                   vmax=err_v_don.max()*100, cmap='cmo.amp', shading='gouraud')
axs[4,2].axis('off')
axs[4,2].set_xlim([-72.50, -72.46])
axs[4,2].set_ylim([40.82,40.88]) 

axs[0, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.05, 0.5, 'M-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)
axs[3, 0].text(-0.05, 0.5, 'L-DeepONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[3,0].transAxes)
axs[4, 0].text(-0.05, 0.5, 'MIONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[4,0].transAxes)

axs[0, 0].set_title('Water Surface Elevation (r: 0.1)', size=20,transform=axs[0,0].transAxes)
axs[0, 1].set_title('U-Velocity (r: 0.0025)', size=20,transform=axs[0,1].transAxes)
axs[0, 2].set_title('V-Velocity (r: 0.0025)', size=20,transform=axs[0,2].transAxes)


cbar = plt.colorbar(last_h, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)',  size=20) 